# HuggingFace Backup - CaML

Backs up models, datasets, and spaces from `CompassioninMachineLearning` to `Backup-CaML`.

**Colab free tier:** ~100GB disk space (enough for models up to ~45GB)

In [ ]:
# Check disk space and cleanup old temp files
!df -h /
!rm -rf /tmp/tmp* 2>/dev/null && echo 'Cleaned up old temp files'
!nvidia-smi || echo 'No GPU (fine for backup)'

In [ ]:
# Install dependencies
!pip install -q huggingface_hub tqdm

In [ ]:
# Authenticate with HuggingFace
# You'll need a token with write access to Backup-CaML
from huggingface_hub import login
login()  # Will prompt for token

In [ ]:
# Verify access
from huggingface_hub import HfApi

api = HfApi()
user = api.whoami()
print(f"Logged in as: {user['name']}")

orgs = {org['name']: org.get('roleInOrg', 'unknown') for org in user.get('orgs', [])}
print(f"CompassioninMachineLearning: {orgs.get('CompassioninMachineLearning', 'NOT FOUND')}")
print(f"Backup-CaML: {orgs.get('Backup-CaML', 'NOT FOUND')}")

In [ ]:
# Configuration
SOURCE_ORG = "CompassioninMachineLearning"
BACKUP_ORG = "Backup-CaML"

In [ ]:
# Core backup functions
import tempfile
import shutil
import glob
import time
import re
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.utils import RepositoryNotFoundError
import os


def cleanup_temp_dirs():
    """Clean up old temp directories from previous failed runs."""
    cleaned = 0
    for pattern in ["/tmp/tmp*", "/var/tmp/tmp*"]:
        for path in glob.glob(pattern):
            if os.path.isdir(path):
                try:
                    shutil.rmtree(path)
                    cleaned += 1
                except Exception:
                    pass
    return cleaned


def get_repo_files(api, repo_id, repo_type="model", max_retries=3):
    """Get dict of {filename: sha} for a repo, or None if not found.

    Uses SHA hashes for content comparison:
    - LFS files (model weights): lfs.sha256
    - Non-LFS files (configs, READMEs): blob_id (git blob SHA)
    """
    for attempt in range(max_retries):
        try:
            if repo_type == "model":
                info = api.model_info(repo_id, files_metadata=True)
            elif repo_type == "dataset":
                info = api.dataset_info(repo_id, files_metadata=True)
            else:  # space
                info = api.space_info(repo_id, files_metadata=True)
            if info.siblings:
                result = {}
                for f in info.siblings:
                    # LFS files have sha256 in lfs attribute, others use blob_id
                    if f.lfs:
                        sha = f.lfs.get("sha256")
                    else:
                        sha = getattr(f, 'blob_id', None)
                    result[f.rfilename] = sha
                return result
            return {}
        except RepositoryNotFoundError:
            return None
        except Exception as e:
            error_str = str(e)
            # Check for rate limiting (429)
            if "429" in error_str or "rate limit" in error_str.lower():
                # Try to extract wait time from error message
                wait_time = 30  # default
                if "Retry after" in error_str:
                    try:
                        match = re.search(r'Retry after (\d+)', error_str)
                        if match:
                            wait_time = int(match.group(1)) + 1
                    except:
                        pass
                if attempt < max_retries - 1:
                    print(f"⏳ Rate limited for {repo_id}, waiting {wait_time}s (attempt {attempt + 1}/{max_retries})")
                    time.sleep(wait_time)
                    continue
            print(f"⚠️  API error for {repo_id}: {type(e).__name__}: {e}")
            return None
    return None


def get_repo_size(api, repo_id, repo_type="model"):
    """Get total size of a repository in bytes."""
    try:
        if repo_type == "model":
            info = api.model_info(repo_id, files_metadata=True)
        elif repo_type == "dataset":
            info = api.dataset_info(repo_id, files_metadata=True)
        else:  # space
            info = api.space_info(repo_id, files_metadata=True)
        if info.siblings:
            return sum(f.size for f in info.siblings if f.size)
        return 0
    except Exception:
        return 0


def backup_repo(repo_name, repo_type="model", source_org=SOURCE_ORG, backup_org=BACKUP_ORG):
    """Backup a single repo (model, dataset, or space)."""
    api = HfApi()
    source_id = f"{source_org}/{repo_name}"
    backup_id = f"{backup_org}/{repo_name}"
    
    # Get source file info
    source_files = get_repo_files(api, source_id, repo_type)
    if source_files is None:
        print(f"❌ [{repo_type}] {repo_name}: Source not found")
        return False
    
    size_bytes = get_repo_size(api, source_id, repo_type)
    size_str = f"{size_bytes / (1024**3):.2f} GB" if size_bytes > 1024**3 else f"{size_bytes / (1024**2):.2f} MB"
    
    # Check if backup exists AND matches source
    backup_files = get_repo_files(api, backup_id, repo_type)
    needs_clean_sync = False
    
    if backup_files is not None:
        if backup_files == source_files:
            print(f"⏭️  [{repo_type}] {repo_name}: Already backed up correctly ({size_str})")
            return True
        else:
            # Backup exists but doesn't match - need clean sync
            missing = set(source_files.keys()) - set(backup_files.keys())
            extra = set(backup_files.keys()) - set(source_files.keys())
            sha_diff = [f for f in source_files if f in backup_files and source_files[f] != backup_files[f]]
            print(f"⚠️  [{repo_type}] {repo_name}: Backup mismatched, will clean sync...")
            if missing:
                print(f"   Missing files: {len(missing)}")
            if extra:
                print(f"   Extra files: {len(extra)}")
            if sha_diff:
                print(f"   Content mismatches: {len(sha_diff)}")
            needs_clean_sync = True
    
    print(f"📦 [{repo_type}] {repo_name}: {size_str}")
    
    # Check disk space
    free_gb = shutil.disk_usage('/').free / (1024**3)
    size_gb = size_bytes / (1024**3)
    if size_gb > free_gb - 5:  # Keep 5GB buffer
        print(f"❌ Not enough disk space (need {size_gb:.1f}GB, have {free_gb:.1f}GB)")
        return False
    
    try:
        with tempfile.TemporaryDirectory() as tmpdir:
            local_path = Path(tmpdir) / repo_name
            
            # Download
            print(f"   ⬇️  Downloading...")
            snapshot_download(
                repo_id=source_id,
                local_dir=str(local_path),
                repo_type=repo_type,
            )
            
            # Show download complete with size
            downloaded_size = sum(f.stat().st_size for f in local_path.rglob('*') if f.is_file()) / (1024**3)
            print(f"   ⬇️  Downloaded {downloaded_size:.2f} GB")
            
            # Create backup repo (public) if needed
            print(f"   📝 Creating/updating backup repo...")
            api.create_repo(
                repo_id=backup_id,
                repo_type=repo_type,
                exist_ok=True,
                private=False,
            )
            
            # Upload - use delete_patterns="*" for clean sync if mismatched
            print(f"   ⬆️  Uploading {downloaded_size:.2f} GB...")
            if needs_clean_sync:
                print(f"   🧹 Clean sync: deleting old files first...")
                api.upload_folder(
                    folder_path=str(local_path),
                    repo_id=backup_id,
                    repo_type=repo_type,
                    delete_patterns="*",  # Delete all existing files for clean sync
                )
            else:
                api.upload_folder(
                    folder_path=str(local_path),
                    repo_id=backup_id,
                    repo_type=repo_type,
                )
            print(f"   ⬆️  Upload complete")
            
            # Verify upload matches source
            new_backup_files = get_repo_files(api, backup_id, repo_type)
            if new_backup_files == source_files:
                print(f"✅ [{repo_type}] {repo_name}: Backup complete and verified")
                return True
            else:
                print(f"⚠️  [{repo_type}] {repo_name}: Upload completed but verification failed!")
                return False
            
    except Exception as e:
        print(f"❌ [{repo_type}] {repo_name}: Failed - {e}")
        return False


# Convenience wrappers
def backup_model(model_name):
    return backup_repo(model_name, "model")

def backup_dataset(dataset_name):
    return backup_repo(dataset_name, "dataset")

def backup_space(space_name):
    return backup_repo(space_name, "space")

print("Backup functions loaded.")

In [ ]:
# List all repos and their backup status
from huggingface_hub import HfApi
from tqdm import tqdm

api = HfApi()

print(f"Listing all repos in {SOURCE_ORG}...")

# Get all source repos
models = list(api.list_models(author=SOURCE_ORG))
datasets = list(api.list_datasets(author=SOURCE_ORG))
spaces = list(api.list_spaces(author=SOURCE_ORG))

print(f"Found {len(models)} models, {len(datasets)} datasets, {len(spaces)} spaces")

# Get sizes and combine all repos
print("\nGetting sizes...")
all_repos = []

for m in tqdm(models, desc="Models"):
    name = m.id.split('/')[-1]
    size = get_repo_size(api, m.id, "model")
    all_repos.append((name, "model", size))

for d in tqdm(datasets, desc="Datasets"):
    name = d.id.split('/')[-1]
    size = get_repo_size(api, d.id, "dataset")
    all_repos.append((name, "dataset", size))

for s in tqdm(spaces, desc="Spaces"):
    name = s.id.split('/')[-1]
    size = get_repo_size(api, s.id, "space")
    all_repos.append((name, "space", size))

# Sort by size (smallest first for efficient progress)
all_repos.sort(key=lambda x: x[2])

total_gb = sum(r[2] for r in all_repos) / (1024**3)
print(f"\nTotal: {len(all_repos)} repos, {total_gb:.1f} GB")

In [ ]:
# Backup a single repo (test)
# Uncomment and change the name to test

# backup_model("some_model_name")
# backup_dataset("some_dataset_name")
# backup_space("some_space_name")

In [ ]:
# Backup ALL repos sorted by size (smallest first)
# This backs up models, datasets, and spaces in one pass

from tqdm import tqdm
import shutil

# Clean up old temp files first
cleaned = cleanup_temp_dirs()
if cleaned > 0:
    print(f"Cleaned up {cleaned} old temp directories")

# Show initial disk space
free_gb = shutil.disk_usage('/').free / (1024**3)
print(f"Disk free: {free_gb:.1f} GB")
print(f"\nBacking up {len(all_repos)} repos (smallest first)...\n")

# Stats
stats = {
    'models_checked': 0, 'models_backed_up': 0, 'models_skipped': 0, 'models_failed': 0,
    'datasets_checked': 0, 'datasets_backed_up': 0, 'datasets_skipped': 0, 'datasets_failed': 0,
    'spaces_checked': 0, 'spaces_backed_up': 0, 'spaces_skipped': 0, 'spaces_failed': 0,
}

for name, repo_type, size in tqdm(all_repos, desc="Repos"):
    stats[f'{repo_type}s_checked'] += 1
    
    result = backup_repo(name, repo_type)
    
    if result:
        # Check if it was skipped (already exists) or actually backed up
        # For simplicity, count as backed_up (includes skipped)
        stats[f'{repo_type}s_backed_up'] += 1
    else:
        stats[f'{repo_type}s_failed'] += 1
    
    # Clean up temp files after each repo
    cleanup_temp_dirs()
    
    # Show disk space periodically
    free_gb = shutil.disk_usage('/').free / (1024**3)
    print(f"   Disk free: {free_gb:.1f} GB\n")

# Print summary
print("\n" + "=" * 60)
print("BACKUP SUMMARY")
print("=" * 60)
print(f"Models checked:    {stats['models_checked']}")
print(f"Models backed up:  {stats['models_backed_up']}")
print(f"Models failed:     {stats['models_failed']}")
print()
print(f"Datasets checked:  {stats['datasets_checked']}")
print(f"Datasets backed up:{stats['datasets_backed_up']}")
print(f"Datasets failed:   {stats['datasets_failed']}")
print()
print(f"Spaces checked:    {stats['spaces_checked']}")
print(f"Spaces backed up:  {stats['spaces_backed_up']}")
print(f"Spaces failed:     {stats['spaces_failed']}")
print("=" * 60)

In [ ]:
# Verify all backups match source
from huggingface_hub import HfApi
from tqdm import tqdm

api = HfApi()

print("Verifying all backups...\n")

# Check models
backup_models = list(api.list_models(author=BACKUP_ORG))
backup_datasets = list(api.list_datasets(author=BACKUP_ORG))
backup_spaces = list(api.list_spaces(author=BACKUP_ORG))

print(f"Backup org has: {len(backup_models)} models, {len(backup_datasets)} datasets, {len(backup_spaces)} spaces")

correct = 0
mismatch = []
errors = []

def verify_repo(backup_id, repo_type):
    name = backup_id.split("/")[-1]
    source_id = f"{SOURCE_ORG}/{name}"
    
    src_files = get_repo_files(api, source_id, repo_type)
    bak_files = get_repo_files(api, backup_id, repo_type)
    
    if src_files is None:
        return "error", f"Source not found: {source_id}"
    if bak_files is None:
        return "error", f"Backup not found: {backup_id}"
    
    if src_files == bak_files:
        return "correct", None
    else:
        return "mismatch", name

for m in tqdm(backup_models, desc="Verifying models"):
    status, info = verify_repo(m.id, "model")
    if status == "correct":
        correct += 1
    elif status == "mismatch":
        mismatch.append(("model", info))
    else:
        errors.append(("model", info))

for d in tqdm(backup_datasets, desc="Verifying datasets"):
    status, info = verify_repo(d.id, "dataset")
    if status == "correct":
        correct += 1
    elif status == "mismatch":
        mismatch.append(("dataset", info))
    else:
        errors.append(("dataset", info))

for s in tqdm(backup_spaces, desc="Verifying spaces"):
    status, info = verify_repo(s.id, "space")
    if status == "correct":
        correct += 1
    elif status == "mismatch":
        mismatch.append(("space", info))
    else:
        errors.append(("space", info))

print(f"\n=== VERIFICATION RESULTS ===")
print(f"Correct: {correct}")
print(f"Mismatch: {len(mismatch)}")
print(f"Errors: {len(errors)}")

if mismatch:
    print(f"\nMismatched repos:")
    for repo_type, name in mismatch:
        print(f"  [{repo_type}] {name}")

if errors:
    print(f"\nErrors:")
    for repo_type, msg in errors:
        print(f"  [{repo_type}] {msg}")

In [ ]:
# Check disk space
!df -h /